# Nebius Data Lab + Fine-Tuning Walkthrough for Google Colab

This is a **fully standalone** notebook for recording a step-by-step demo.

In this notebook, we run the full app flow one step at a time: generate logs, upload them to Data Lab, improve them with a stronger teacher model, clean the results into training data, fine-tune with LoRA, deploy, and smoke test.

This format is meant for a video demo, so each step has a short explanation you can say out loud while you run it.

## Before you run

### What this notebook teaches
- **Data Lab** is where we store and process datasets.
- A **teacher model** is a stronger model that writes better answers.
- **Distillation** means we use the teacher's answers to create better training data.
- **Fine-tuning** means we train a smaller model to behave like the support bot we want.

### Why we still start with baseline logs
For the app you built, we want to show the **full Data Lab lifecycle**, not only the shortest path. So we first create raw logs, then improve them with a stronger model, and then fine-tune on the improved dataset.

### What you need
- A Nebius API key
- Permission to create Data Lab datasets, batch jobs, fine-tuning jobs, and custom models
- Patience: fine-tuning and deployment can take a while

> Tip: in Colab, run one cell at a time from top to bottom.

In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openai', 'requests'])

0

In [2]:
import getpass
import os

if not os.environ.get('NEBIUS_API_KEY'):
    os.environ['NEBIUS_API_KEY'] = getpass.getpass('Enter your NEBIUS_API_KEY: ')

print('API key loaded:', bool(os.environ.get('NEBIUS_API_KEY')))

Enter your NEBIUS_API_KEY: ··········
API key loaded: True


## Step 0 — Setup and configuration

This is the **control panel** for the whole notebook. If you want to change the prompts, the models, or the adapter name, this is the place to do it.

In [4]:
import json
import time
import uuid
from pathlib import Path
from typing import Any
from urllib.parse import quote

import requests
from openai import OpenAI

API_KEY = os.environ['NEBIUS_API_KEY']
BASE_URL = 'https://api.tokenfactory.nebius.com/v1/'
CONTROL_URL = 'https://api.tokenfactory.nebius.com'

TEACHER_MODEL = 'meta-llama/Llama-3.3-70B-Instruct'
FINE_TUNE_BASE_MODEL = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
DEPLOY_BASE_MODEL_CANDIDATES = [FINE_TUNE_BASE_MODEL, 'meta-llama/Llama-3.1-8B-Instruct']

LORA_RANK = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
N_EPOCHS = 2
POLL_SECONDS = 30
BATCH_COMPLETION_WINDOW = '24h'
DATA_LAB_FOLDER = '/demo/customer-support'
ADAPTER_NAME = f'customer-support-colab-{uuid.uuid4().hex[:6]}'

ARTIFACT_DIR = Path('customer_support_demo_artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
STATE_PATH = ARTIFACT_DIR / 'customer_notebook_state.json'
BASELINE_LOGS_PATH = ARTIFACT_DIR / 'baseline_logs.json'
RAW_DATASET_PATH = ARTIFACT_DIR / 'customer_raw_dataset.jsonl'
RAW_BATCH_OUTPUT_PATH = ARTIFACT_DIR / 'customer_batch_output.jsonl'
CURATED_TRAINING_PATH = ARTIFACT_DIR / 'customer_curated_training.jsonl'

SUPPORT_SYSTEM_PROMPT = (
    'You are a helpful customer support assistant for Northwind Gadgets. '
    'Policies: 30-day returns for unused physical items; damaged items reported within 7 days can be replaced; '
    'refunds go to the original payment method in 5-7 business days after approval; digital gift cards are non-refundable; '
    'subscription cancellations take effect at the end of the current billing cycle; shipping addresses can be changed only before fulfillment; '
    'warranty claims require the product serial number and proof of purchase; expedited shipping fees are non-refundable once an order has shipped.'
)

TEACHER_SYSTEM_PROMPT = (
    'You are a senior customer support policy expert. '
    'Answer clearly, empathetically, and accurately. '
    'Never promise exceptions to policy. Ask only for the minimum extra information required to proceed.'
)

DOMAIN_TOPICS = [
    'My headphones arrived yesterday and the right earcup is cracked. What can you do?',
    'I bought a travel backpack 12 days ago and never used it. Can I return it for a refund?',
    'I accidentally sent a digital gift card to the wrong email. Can you refund it?',
    'Can I change the shipping address on my order if it still says preparing for shipment?',
    'Please cancel my premium support subscription today. I do not want to be billed again.',
    'Tracking shows my return was delivered four days ago, but I still do not see the refund.',
    'My espresso machine stopped heating after two months. Can I file a warranty claim?',
    'I paid for expedited shipping, but I want that charge refunded after the order shipped.',
    'My monitor already shipped this morning. Can you reroute it to my office instead?',
    'My refund was approved, but the card I used is closed. Can you send it to a new card?',
]

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
HEADERS = {'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'}

print('Artifacts folder:', ARTIFACT_DIR.resolve())
print('Adapter name for this run:', ADAPTER_NAME)
print('Prompts in demo set:', len(DOMAIN_TOPICS))

Artifacts folder: /content/customer_support_demo_artifacts
Adapter name for this run: customer-support-colab-dbc4e3
Prompts in demo set: 10


## Helper functions

This cell contains the reusable functions for the whole workflow. You only need to run it once. After that, each step below becomes a small, easy-to-follow action.

In [6]:
def load_state() -> dict[str, Any]:
    if not STATE_PATH.exists():
        return {}
    return json.loads(STATE_PATH.read_text())

def save_state(state: dict[str, Any]) -> None:
    STATE_PATH.write_text(json.dumps(state, indent=2, sort_keys=True) + '\n')

def datalab_request(method: str, path: str, *, json_body: dict[str, Any] | None = None, params: dict[str, Any] | None = None) -> requests.Response:
    response = requests.request(method, f'{CONTROL_URL}{path}', headers=HEADERS, json=json_body, params=params, timeout=60)
    response.raise_for_status()
    return response

def teacher_messages(prompt: str) -> list[dict[str, str]]:
    return [
        {'role': 'system', 'content': TEACHER_SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]

def extract_text_content(value: Any) -> str:
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        parts = [extract_text_content(item) for item in value]
        return '\n'.join([part for part in parts if part]).strip()
    if isinstance(value, dict):
        if isinstance(value.get('text'), str):
            return value['text'].strip()
        for key in ('content', 'message'):
            text = extract_text_content(value.get(key))
            if text:
                return text
    return ''

def extract_prompt_from_record(record: dict[str, Any], id_map: dict[str, str]) -> str:
    custom_id = str(record.get('custom_id') or '').strip()
    if custom_id and custom_id in id_map:
        return id_map[custom_id]
    for key in ('messages', 'completed_dialogue', 'prompt'):
        value = record.get(key)
        if isinstance(value, list):
            for message in value:
                if isinstance(message, dict) and message.get('role') == 'user':
                    text = extract_text_content(message.get('content'))
                    if text:
                        return text
        elif isinstance(value, str) and value.strip():
            return value.strip()
    return ''

def extract_reply_from_record(record: dict[str, Any]) -> str:
    for key in ('completion', 'assistant_response', 'output_text', 'text'):
        text = extract_text_content(record.get(key))
        if text:
            return text
    response = record.get('response')
    if isinstance(response, dict):
        body = response.get('body') if isinstance(response.get('body'), dict) else response
        choices = body.get('choices') if isinstance(body, dict) else None
        if isinstance(choices, list) and choices:
            choice = choices[0]
            if isinstance(choice, dict):
                message = choice.get('message') if isinstance(choice.get('message'), dict) else None
                if message:
                    text = extract_text_content(message.get('content'))
                    if text:
                        return text
                text = extract_text_content(choice.get('text'))
                if text:
                    return text
        output = body.get('output') if isinstance(body, dict) else None
        if isinstance(output, list):
            for item in output:
                if isinstance(item, dict):
                    text = extract_text_content(item.get('content'))
                    if text:
                        return text
    for key in ('completed_dialogue', 'messages'):
        value = record.get(key)
        if isinstance(value, list):
            assistant_messages = [extract_text_content(message.get('content')) for message in value if isinstance(message, dict) and message.get('role') == 'assistant']
            assistant_messages = [msg for msg in assistant_messages if msg]
            if assistant_messages:
                return assistant_messages[-1]
    return ''

def load_baseline_logs() -> list[dict[str, Any]]:
    if not BASELINE_LOGS_PATH.exists():
        raise RuntimeError('Baseline logs file not found. Run Step 1 first.')
    return json.loads(BASELINE_LOGS_PATH.read_text())

def load_id_map_from_artifact() -> dict[str, str]:
    if not RAW_DATASET_PATH.exists():
        raise RuntimeError('Raw dataset artifact not found. Run Step 2 first.')
    id_map = {}
    with RAW_DATASET_PATH.open() as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            custom_id = record.get('custom_id')
            prompt = record.get('prompt')
            if isinstance(custom_id, str) and isinstance(prompt, str):
                id_map[custom_id] = prompt
    return id_map

def step1_generate_baseline_logs(topics: list[str]) -> list[dict[str, Any]]:
    logs = []
    for topic in topics:
        resp = client.chat.completions.create(model=FINE_TUNE_BASE_MODEL, messages=[{'role': 'system', 'content': SUPPORT_SYSTEM_PROMPT}, {'role': 'user', 'content': topic}], max_tokens=256)
        logs.append({'prompt': topic, 'completion': resp.choices[0].message.content, 'model': FINE_TUNE_BASE_MODEL})
        print('✓', topic[:75])
    BASELINE_LOGS_PATH.write_text(json.dumps(logs, indent=2))
    return logs

def step2_upload_raw_dataset(logs: list[dict[str, Any]]) -> tuple[str, str, dict[str, str]]:
    dataset_name = f'customer-support-raw-{uuid.uuid4().hex[:8]}'
    rows, id_map = [], {}
    with RAW_DATASET_PATH.open('w') as handle:
        for entry in logs:
            record = {'custom_id': uuid.uuid4().hex, 'prompt': entry['prompt'], 'messages': teacher_messages(entry['prompt']), 'base_completion': entry['completion'] or '', 'base_model': entry['model']}
            handle.write(json.dumps(record) + '\n')
            rows.append(record)
            id_map[record['custom_id']] = entry['prompt']
    payload = {'name': dataset_name, 'schema': [{'name': 'custom_id', 'type': {'name': 'string'}}, {'name': 'prompt', 'type': {'name': 'string'}}, {'name': 'messages', 'type': {'name': 'json'}}, {'name': 'base_completion', 'type': {'name': 'string'}}, {'name': 'base_model', 'type': {'name': 'string'}}], 'folder': DATA_LAB_FOLDER, 'rows': rows}
    dataset = datalab_request('POST', '/v1/datasets', json_body=payload).json()
    return dataset['id'], dataset['current_version'], id_map

def step3_run_batch_inference(source_dataset_id: str, source_dataset_version: str) -> tuple[str, str]:
    payload = {'type': 'batch_inference', 'src': [{'id': source_dataset_id, 'version': source_dataset_version, 'mapping': {'type': 'text_messages', 'messages': {'type': 'column', 'name': 'messages'}, 'custom_id': {'type': 'column', 'name': 'custom_id'}, 'max_tokens': {'type': 'text', 'value': '1024'}}}], 'dst': [], 'params': {'model': TEACHER_MODEL, 'completion_window': BATCH_COMPLETION_WINDOW}}
    operation = datalab_request('POST', '/v1/operations', json_body=payload).json()
    if not operation.get('dst'):
        raise RuntimeError(f'No destination dataset returned: {json.dumps(operation, indent=2)}')
    op_id = operation['id']
    dst_dataset_id = operation['dst'][0]['id']
    print('Operation created:', op_id)
    print('Output dataset ID:', dst_dataset_id)
    while True:
        time.sleep(POLL_SECONDS)
        operation = datalab_request('GET', f'/v1/operations/{op_id}').json()
        status = operation['status']
        print('Batch status:', status)
        if status in {'succeeded', 'failed', 'cancelled', 'unknown'}:
            break
    if status != 'succeeded':
        raise RuntimeError(f'Batch inference ended with status={status}')
    return op_id, dst_dataset_id

def step4_download_and_curate(output_dataset_id: str, id_map: dict[str, str]) -> Path:
    export_response = datalab_request('GET', f'/v1/datasets/{output_dataset_id}/export', params={'format': 'jsonl'})
    RAW_BATCH_OUTPUT_PATH.write_text(export_response.text)
    records = [json.loads(line) for line in export_response.text.strip().splitlines() if line.strip()]
    written = 0
    with CURATED_TRAINING_PATH.open('w') as handle:
        for rec in records:
            prompt = extract_prompt_from_record(rec, id_map)
            reply = extract_reply_from_record(rec)
            if not prompt or not reply or len(reply) < 50:
                continue
            sample = {'messages': [{'role': 'system', 'content': SUPPORT_SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}, {'role': 'assistant', 'content': reply}]}
            handle.write(json.dumps(sample) + '\n')
            written += 1
    if not records:
        raise RuntimeError('Batch output export returned no rows.')
    if written == 0:
        raise RuntimeError('No usable assistant replies were found in the teacher output.')
    print('Curated examples written:', written)
    return CURATED_TRAINING_PATH

def step5_upload_training_file(curated_path: Path) -> str:
    with curated_path.open('rb') as handle:
        file_obj = client.files.create(file=handle, purpose='fine-tune')
    return file_obj.id

def step6_launch_finetuning(train_file_id: str) -> str:
    job = client.fine_tuning.jobs.create(model=FINE_TUNE_BASE_MODEL, training_file=train_file_id, hyperparameters={'learning_rate': 2e-5, 'n_epochs': N_EPOCHS, 'lora': True, 'lora_r': LORA_RANK, 'lora_alpha': LORA_ALPHA, 'lora_dropout': LORA_DROPOUT, 'packing': True})
    print('Fine-tune job:', job.id)
    while True:
        time.sleep(POLL_SECONDS)
        job = client.fine_tuning.jobs.retrieve(job.id)
        print('Fine-tune status:', job.status, 'trained_tokens:', getattr(job, 'trained_tokens', 0))
        if job.status in {'succeeded', 'failed', 'cancelled'}:
            break
    if job.status != 'succeeded':
        error_text = getattr(job, 'error', '')
        raise RuntimeError(f'Fine-tuning failed: {error_text}')
    return job.id

def step7_deploy_lora(ft_job_id: str, adapter_name: str) -> dict[str, Any]:
    checkpoints = client.fine_tuning.jobs.checkpoints.list(ft_job_id).data
    if not checkpoints:
        raise RuntimeError('No checkpoints found for this job.')
    ckpt_id = checkpoints[-1].id
    attempt_errors, model_info = [], None
    for base_model in DEPLOY_BASE_MODEL_CANDIDATES:
        payload = {'source': f'{ft_job_id}:{ckpt_id}', 'base_model': base_model, 'name': adapter_name, 'description': 'Standalone Colab customer-support demo model'}
        response = requests.post(f'{CONTROL_URL}/v0/models', json=payload, headers=HEADERS, timeout=60)
        if response.ok:
            model_info = response.json()
            print('Deploy request accepted with base model:', base_model)
            break
        body_text = response.text.strip() or '<empty body>'
        attempt_errors.append(f'base_model={base_model} status={response.status_code} body={body_text}')
        print('Deploy rejected for base model:', base_model)
    if model_info is None:
        raise RuntimeError('LoRA deployment failed for all candidate base models:\n- ' + '\n- '.join(attempt_errors))
    model_name = model_info['name']
    print('Deploy initiated:', model_name)
    for _ in range(60):
        time.sleep(10)
        encoded_name = quote(model_name, safe='')
        response = requests.get(f'{CONTROL_URL}/v0/models/{encoded_name}', headers=HEADERS, timeout=60)
        response.raise_for_status()
        info = response.json()
        status = info.get('status', '?')
        print('Deploy status:', status)
        if status == 'active':
            return info
        if status == 'error':
            reason = info.get('status_reason')
            raise RuntimeError(f'Deployment failed: {reason}')
    raise TimeoutError('Model did not become active in time.')

def step8_smoke_test(model_full_name: str) -> list[dict[str, str]]:
    test_prompts = ['My package arrived damaged yesterday. What should I do?', 'Can I return an unused backpack after 12 days?']
    answers = []
    for prompt in test_prompts:
        resp = client.chat.completions.create(model=model_full_name, messages=[{'role': 'system', 'content': SUPPORT_SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}], max_tokens=200)
        answer = resp.choices[0].message.content
        answers.append({'prompt': prompt, 'answer': answer})
        print('Q:', prompt)
        print('A:', answer)
        print()
    return answers

print('Helper functions loaded.')

Helper functions loaded.


## Step 1 — Generate baseline customer-support logs

### What we are doing
We ask a smaller support model to answer a list of customer questions.

### Why we do it
This creates the **raw logs** for our app. In a real system, these could come from an existing chatbot or support assistant already in production.

### In simple words
This is our **first draft dataset**.

In [7]:
logs = step1_generate_baseline_logs(DOMAIN_TOPICS)
state = load_state()
state.update({'last_completed_step': 1, 'baseline_log_count': len(logs)})
save_state(state)
print('Saved baseline logs to:', BASELINE_LOGS_PATH)
logs[:2]

✓ My headphones arrived yesterday and the right earcup is cracked. What can y
✓ I bought a travel backpack 12 days ago and never used it. Can I return it f
✓ I accidentally sent a digital gift card to the wrong email. Can you refund 
✓ Can I change the shipping address on my order if it still says preparing fo
✓ Please cancel my premium support subscription today. I do not want to be bi
✓ Tracking shows my return was delivered four days ago, but I still do not se
✓ My espresso machine stopped heating after two months. Can I file a warranty
✓ I paid for expedited shipping, but I want that charge refunded after the or
✓ My monitor already shipped this morning. Can you reroute it to my office in
✓ My refund was approved, but the card I used is closed. Can you send it to a
Saved baseline logs to: customer_support_demo_artifacts/baseline_logs.json


[{'prompt': 'My headphones arrived yesterday and the right earcup is cracked. What can you do?',
  'completion': "Sorry to hear that your headphones didn't arrive in the condition you expected.\n\nSince you reported the issue within 7 days of delivery, we can certainly help you get it replaced. To facilitate the process, I'll need you to provide me with a few details:\n\n1. Your order number, which can be found in the order confirmation email you received.\n2. A clear photo of the cracked right earcup, so I can see the damage.\n3. Your shipping address, so we can send the replacement headphones to you as soon as possible.\n\nPlease go ahead and provide this information, and I'll process the replacement right away.\n\nIf you have any questions or need further assistance, feel free to ask!",
  'model': 'meta-llama/Meta-Llama-3.1-8B-Instruct'},
 {'prompt': 'I bought a travel backpack 12 days ago and never used it. Can I return it for a refund?',
  'completion': "According to our return po

## Step 2 — Upload the raw logs to Data Lab

### What we are doing
We store the baseline logs as a structured dataset in Data Lab.

### Why we do it
Once the data is in Data Lab, it becomes easier to version, inspect, reuse, and pass into batch inference jobs.

### In simple words
We are moving our rough notes into an organized online workbook.

In [8]:
if 'logs' not in globals():
    logs = load_baseline_logs()

raw_dataset_id, raw_dataset_version, id_map = step2_upload_raw_dataset(logs)
state = load_state()
state.update({'raw_dataset_id': raw_dataset_id, 'raw_dataset_version': raw_dataset_version, 'last_completed_step': 2})
save_state(state)
print('Raw Data Lab dataset ID:', raw_dataset_id)
print('Raw Data Lab dataset version:', raw_dataset_version)

Raw Data Lab dataset ID: 7d40a55252364131b4e9e91d7bd113aa
Raw Data Lab dataset version: e1c550718899483a99479773b05a9d05


## Step 3 — Run batch inference with a stronger teacher model

### What we are doing
We send the customer questions to a **stronger model**. This stronger model writes better customer-support answers.

### Why we do it
We want the final training data to be better than our rough first draft.

### In simple words
The **teacher** is correcting and improving the rough answers.

> This step can take a while because batch inference runs asynchronously.

In [ ]:
if 'raw_dataset_id' not in globals() or 'raw_dataset_version' not in globals():
    state = load_state()
    raw_dataset_id = state['raw_dataset_id']
    raw_dataset_version = state['raw_dataset_version']

operation_id, output_dataset_id = step3_run_batch_inference(raw_dataset_id, raw_dataset_version)
state = load_state()
state.update({'batch_operation_id': operation_id, 'output_dataset_id': output_dataset_id, 'last_completed_step': 3})
save_state(state)
print('Batch operation ID:', operation_id)
print('Teacher output dataset ID:', output_dataset_id)

Operation created: batch__019ce163-0775-7d12-9eb0-de15bec3e945
Output dataset ID: 54a37f68cc704702b29d83209a789b42
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Batch status: queued
Bat

## Step 4 — Download and clean the teacher outputs

### What we are doing
We export the teacher-generated dataset, recover the original prompt for each row, keep strong answers, and write the final chat format needed for fine-tuning.

### Why we do it
Not every machine-generated answer is worth training on. We clean the dataset so the student model learns from the best examples.

### In simple words
We are turning the teacher's corrections into a neat study guide.

In [ ]:
if 'output_dataset_id' not in globals():
    output_dataset_id = load_state()['output_dataset_id']
if 'id_map' not in globals():
    id_map = load_id_map_from_artifact()

curated_path = step4_download_and_curate(output_dataset_id, id_map)
state = load_state()
state.update({'curated_path': str(curated_path), 'last_completed_step': 4})
save_state(state)
print('Curated training file saved at:', curated_path)

## Step 5 — Upload the curated training file

### What we are doing
We upload the clean JSONL file to the Nebius fine-tuning files API.

### Why we do it
Fine-tuning jobs train from uploaded files with `purpose='fine-tune'`, not directly from a Data Lab export.

### In simple words
We are handing the student the final cleaned workbook.

In [ ]:
if 'curated_path' not in globals():
    curated_path = Path(load_state()['curated_path'])

training_file_id = step5_upload_training_file(curated_path)
state = load_state()
state.update({'training_file_id': training_file_id, 'last_completed_step': 5})
save_state(state)
print('Training file ID:', training_file_id)

## Step 6 — Fine-tune the support model with LoRA

### What we are doing
We launch a fine-tuning job so the smaller model learns from the cleaned teacher-generated examples.

### Why we do it
This turns a general-purpose model into a customer-support assistant that better follows your policy and style.

### What LoRA means in plain English
LoRA is a lightweight way to train a model without changing every single weight. It is cheaper and faster than full fine-tuning.

In [ ]:
if 'training_file_id' not in globals():
    training_file_id = load_state()['training_file_id']

ft_job_id = step6_launch_finetuning(training_file_id)
state = load_state()
state.update({'fine_tune_job_id': ft_job_id, 'last_completed_step': 6})
save_state(state)
print('Fine-tuning job ID:', ft_job_id)

## Step 7 — Deploy the fine-tuned model

### What we are doing
We take the latest checkpoint from the fine-tuning job and deploy it as a private custom model.

### Why we do it
Training alone is not enough. Deployment turns the result into something you can actually call with an API.

### In simple words
This is the moment where the trained model becomes a usable application.

In [ ]:
if 'ft_job_id' not in globals():
    ft_job_id = load_state()['fine_tune_job_id']

model_info = step7_deploy_lora(ft_job_id, ADAPTER_NAME)
deployed_model_name = model_info['name']
state = load_state()
state.update({'deployed_model_name': deployed_model_name, 'adapter_name': ADAPTER_NAME, 'last_completed_step': 7})
save_state(state)
print('Deployed model name:', deployed_model_name)

## Step 8 — Smoke test the deployed model

### What we are doing
We ask the deployed support model a couple of customer questions.

### Why we do it
This is a quick reality check. We want to see that the model is alive, helpful, and roughly following policy.

### In simple words
This is the first mini-exam for the model.

In [ ]:
if 'deployed_model_name' not in globals():
    deployed_model_name = load_state()['deployed_model_name']

smoke_test_answers = step8_smoke_test(deployed_model_name)
state = load_state()
state.update({'last_completed_step': 8})
save_state(state)
smoke_test_answers

## What you can say in your video

### Quick storytelling version
- We started with rough customer-support logs.
- We uploaded them to Data Lab.
- We used a stronger teacher model to improve the answers.
- We cleaned those improved answers into a training dataset.
- We fine-tuned a smaller model with LoRA.
- We deployed it.
- We tested it like a real support assistant.

### Troubleshooting tips
- If your kernel restarts, rerun the setup and helper cells, then continue using `load_state()`.
- If deployment fails, use a fresh adapter name and check whether the fine-tuning job really succeeded.
- The teacher output dataset in Data Lab is **not** the final training file. The curation step is what turns it into fine-tuning-ready JSONL.

### Most important beginner idea
This notebook shows **distillation + fine-tuning**, not only fine-tuning. The teacher creates better examples, and then the student model learns from those examples.